In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 24.72 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Dataset Verification, Head Inspection

import os
import glob
import pandas as pd

# 1. Locate dataset CSV
possible_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert len(possible_csv) > 0, "Could not find Data_Entry_2017.csv in /kaggle/input/"
csv_path = possible_csv[0]
print(f"Found Metadata CSV: {csv_path}")

df = pd.read_csv(csv_path)

# 2. Map image index to full filesystem path across all image folders
all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

# 3. 14 standard thorax pathologies in NIH-14
CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

# 4. Official train/test split using test_list.txt
test_list_files = glob.glob('/kaggle/input/**/test_list.txt', recursive=True)
assert len(test_list_files) > 0, "Could not find test_list.txt"
with open(test_list_files[0]) as f:
    test_set_names = set(line.strip() for line in f)

train_df = df[~df['Image Index'].isin(test_set_names)].reset_index(drop=True)
test_df = df[df['Image Index'].isin(test_set_names)].reset_index(drop=True)

print(f"\n✓ Verified Images Available: {len(df)}")
print(f"✓ Train Pool: {len(train_df)} scans | Test Pool: {len(test_df)} scans")

# Display the head of the processed dataset
df[['Image Index', 'Finding Labels'] + CLASSES[:5]].head(5)

Found Metadata CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv

✓ Verified Images Available: 112120
✓ Train Pool: 86524 scans | Test Pool: 25596 scans


,Image Index,Finding Labels,Atelectasis,Cardiomegaly,Effusion,Infiltration,Mass
0,00000001_000.png,Cardiomegaly,0,1,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,0,1,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,0,1,1,0,0
3,00000002_000.png,No Finding,0,0,0,0,0
4,00000003_000.png,Hernia,0,0,0,0,0


In [3]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [4]:
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Transform pipeline matching CheXFound pretraining
eval_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Balanced fast subset: 3,000 train, 1,000 test
train_sample = train_df.sample(3000, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(1000, random_state=42).reset_index(drop=True)

class HybridFeatureDataset(Dataset):
    def __init__(self, data_df, transform):
        self.paths = data_df['full_path'].values
        self.labels = data_df[CLASSES].values.astype('float32')
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.labels[idx])

train_loader = DataLoader(HybridFeatureDataset(train_sample, eval_transform), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(HybridFeatureDataset(test_sample, eval_transform), batch_size=32, shuffle=False, num_workers=2)

# 3. Hybrid Feature Extraction: Concatenates CLS + Mean-Pooled Spatial Patches across 4 blocks
def extract_hybrid_features(loader, backbone, desc="Extracting"):
    all_feats, all_targets = [], []
    backbone.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc):
            imgs = imgs.to(device)
            # intermediate_layers returns [(patch_tokens, cls_token), ...] for each of n=4 blocks
            layers = backbone.get_intermediate_layers(imgs, n=4, return_class_token=True)
            
            block_representations = []
            for patch_tokens, cls_token in layers:
                # patch_tokens: [B, 1024, 1024], cls_token: [B, 1024]
                mean_patch = patch_tokens.mean(dim=1)  # Spatial global average pool: [B, 1024]
                block_representations.extend([cls_token, mean_patch])
            
            # 4 blocks * (1024 CLS + 1024 Spatial Pool) = 8,192-dim hybrid vector
            hybrid_tokens = torch.cat(block_representations, dim=-1)
            all_feats.append(hybrid_tokens.cpu())
            all_targets.append(labels)
            
    return torch.cat(all_feats, dim=0).numpy(), torch.cat(all_targets, dim=0).numpy()

print("--- Extracting Hybrid (CLS + Spatial Patch) Representations ---")
X_tr, y_tr = extract_hybrid_features(train_loader, model, desc="Train Set (3,000)")
X_te, y_te = extract_hybrid_features(test_loader, model, desc="Test Set (1,000)")

print(f"\nExtracted Feature Shapes: X_train {X_tr.shape} | X_test {X_te.shape}")

# 4. Train Independent Calibrated Linear Probes per Disease
print("\n" + "="*58)
print(f"{'Pathology':<22} | {'AUROC':<10} | {'AUPRC':<10}")
print("="*58)

aucs, auprcs = [], []
test_preds = np.zeros_like(y_te)

for i, c in enumerate(CLASSES):
    # Train dedicated balanced logistic classifier with L2 regularizer
    clf = LogisticRegression(max_iter=600, C=0.05, solver='lbfgs', class_weight='balanced')
    clf.fit(X_tr, y_tr[:, i])
    
    probs = clf.predict_proba(X_te)[:, 1]
    test_preds[:, i] = probs
    
    if len(np.unique(y_te[:, i])) > 1:
        roc = roc_auc_score(y_te[:, i], probs)
        prc = average_precision_score(y_te[:, i], probs)
        aucs.append(roc)
        auprcs.append(prc)
        print(f"{c:<22} | {roc:<10.4f} | {prc:<10.4f}")
    else:
        print(f"{c:<22} | {'N/A':<10} | {'N/A':<10}")

mean_auc = np.mean(aucs)
mean_auprc = np.mean(auprcs)
print("-"*58)
print(f"{'Mean Baseline':<22} | {mean_auc:<10.4f} | {mean_auprc:<10.4f}")
print("="*58)

# 5. Save the baseline probe predictions to disk
results_df = pd.DataFrame(test_preds, columns=[f"pred_{c}" for c in CLASSES])
for i, c in enumerate(CLASSES):
    results_df[f"true_{c}"] = y_te[:, i]
results_df.to_csv("/kaggle/working/chexfound_step4_baseline_results.csv", index=False)
print("Saved baseline test predictions to /kaggle/working/chexfound_step4_baseline_results.csv")

--- Extracting Hybrid (CLS + Spatial Patch) Representations ---


Test Set (1,000): 100%|██████████| 32/32 [04:59<00:00,  9.34s/it]



Extracted Feature Shapes: X_train (3000, 8192) | X_test (1000, 8192)

Pathology              | AUROC      | AUPRC     
Atelectasis            | 0.5971     | 0.1711    
Cardiomegaly           | 0.6300     | 0.0781    
Effusion               | 0.6283     | 0.2451    
Infiltration           | 0.5950     | 0.2890    
Mass                   | 0.5394     | 0.0794    
Nodule                 | 0.4756     | 0.0627    
Pneumonia              | 0.4274     | 0.0164    
Pneumothorax           | 0.6133     | 0.2195    
Consolidation          | 0.5628     | 0.0728    
Edema                  | 0.7027     | 0.0787    
Emphysema              | 0.5667     | 0.0608    
Fibrosis               | 0.6970     | 0.0905    
Pleural_Thickening     | 0.6317     | 0.0671    
Hernia                 | 0.7728     | 0.0044    
----------------------------------------------------------
Mean Baseline          | 0.6029     | 0.1097    
Saved baseline test predictions to /kaggle/working/chexfound_step4_baseline_results.cs